In [ ]:
!pip install gradio_client==0.9.0 websockets==11.0.3 -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 306.8/306.8 kB 8.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 118.1/118.1 kB 9.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
langgraph-sdk 0.4.2 requires websockets<16,>=14, but you have websockets 11.0.3 which is incompatible.
gradio 5.50.0 requires gradio-client==1.14.0, but you have gradio-client 0.9.0 which is incompatible.
google-genai 1.68.0 requires websockets<17.0,>=13.0.0, but you have websockets 11.0.3 which is incompatible.
yfinance 0.2.66 requires websockets>=13.0, but you have websockets 11.0.3 which is incompatible.
langsmith 0.8.9 requires websockets>=15.0, but you have websockets 11.0.3 which is incompatible.
dataproc-spark-connect 1.1.0 requires websockets>=14.0, but you have websockets 11.0.3 which is incompatible.
google-adk 1.29.0 requires websockets<16.0.0,>=

In [ ]:
import pandas as pd

In [ ]:
from gradio_client import Client
import tempfile, os, pandas as pd
from pathlib import Path
from google.colab import drive

drive.mount('/content/drive')
df = pd.read_parquet("/content/drive/MyDrive/data.parquet")

client = Client("seungheondoh/LP-Music-Caps-demo")

row = df.iloc[0]
audio_bytes = row['audio']['bytes']
suffix = Path(row['audio'].get('path', 'track.mp3')).suffix or '.mp3'

tmp = tempfile.NamedTemporaryFile(delete=False, suffix=suffix)
tmp.write(audio_bytes)
tmp.close()

# В версии 0.9.0 передаём путь напрямую
result = client.predict(tmp.name, api_name="/predict")
os.unlink(tmp.name)
print(result)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loaded as API: https://seungheondoh-lp-music-caps-demo.hf.space ✔
[0:00-10:00]
This is a hip-hop music piece. There is a male vocalist singing melodically in the lead. The melody is being played by the electric guitar and the keyboard while the bass guitar is playing in the background. The rhythm is provided by an electronic drum beat. The atmosphere is hip and urban. This piece could be used in the soundtrack of a crime movie that takes place in the big city. 
 
[10:00-20:00]
This is a hip-hop music piece. There is a male vocalist singing melodically in the lead. The melody is being played by the electric guitar and the keyboard while the bass guitar is playing in the background. The rhythm is provided by an electronic drum beat. The atmosphere is hip and urban. This piece could be used in the soundtrack of a crime movie that takes place in the big city. 
 



In [ ]:
import time
from tqdm import tqdm

def generate_caption(audio_value, client, retries=3):
    suffix = Path(audio_value.get('path', 'track.mp3')).suffix or '.mp3'
    tmp = tempfile.NamedTemporaryFile(delete=False, suffix=suffix)
    tmp.write(audio_value['bytes'])
    tmp.close()

    raw = None
    for attempt in range(retries):
        try:
            raw = client.predict(tmp.name, api_name="/predict")
            break
        except Exception as e:
            print(f"  Попытка {attempt+1} не удалась: {e}")
            time.sleep(3)

    os.unlink(tmp.name)

    if raw is None:
        return {'full_text': None, 'unique_sentences': None}

    return parse_caption(raw)


In [ ]:
def generate_captions_batch(df, client, audio_col='audio', save_path=None, start_idx=0):
    """
    Генерирует описания для всех треков в датафрейме.
    Сохраняет результат на Google Drive после каждого трека.
    start_idx — с какого индекса продолжить если прервалось
    """
    results = []

    # Если файл уже есть — загружаем прогресс
    if save_path and os.path.exists(save_path):
        existing = pd.read_csv(save_path)
        results = existing.to_dict('records')
        start_idx = len(results)
        print(f"Продолжаем с {start_idx} трека")

    for idx in tqdm(range(start_idx, len(df)), desc="Генерация описаний"):
        row = df.iloc[idx]
        caption = generate_caption(row[audio_col], client)

        results.append({
            'idx': idx,
            'caption': caption
        })

        # Сохраняем после каждого трека
        if save_path:
            pd.DataFrame(results).to_csv(save_path, index=False)

    return pd.DataFrame(results)

In [ ]:
print(generate_caption(df.iloc[67]['audio'], client))

[0:00-10:00]
This song features a male voice singing the main melody. This is accompanied by the percussion playing a simple beat. The hi-hat is played in eighth notes. The bass plays the root notes of the chords. There is no percussion in this song. This song can be played in a romantic movie. 
 
[10:00-20:00]
This song features a male voice singing the main melody. This is accompanied by programmed percussion playing a simple beat. The hi-hat is played in eighth notes. A synth plays chords in the background. There is no percussion in this song. This song can be played in a romantic movie. 
 
[20:00-30:00]
This song features a male voice singing the main melody. This is accompanied by a guitar playing arpeggiated chords. A wind instrument plays fills in between lines. There is no percussion in this song. This song can be played in a romantic movie. 
 



In [ ]:
import re

def parse_caption(raw_caption):
    """
    Из сырого вывода LP-MusicCaps делает:
    1. full_text — весь текст целиком
    2. unique_sentences — упорядоченное множество уникальных предложений
    """
    # Убираем метки времени типа [0:00-10:00]
    text = re.sub(r'\[\d+:\d+-\d+:\d+\]\s*', '', raw_caption)

    # Разбиваем на предложения
    sentences = [s.strip() for s in re.split(r'(?<=[.!?])\s+', text) if s.strip()]

    # Упорядоченное множество — убираем дубли, сохраняем порядок первого появления
    seen = set()
    unique = []
    for s in sentences:
        if s not in seen:
            seen.add(s)
            unique.append(s)

    return {
        'full_text': text.strip(),
        'unique_sentences': ' '.join(unique)
    }


# Тест на примере из твоего вывода
sample = """[0:00-10:00]
This song features a male voice singing the main melody. This is accompanied by the percussion playing a simple beat. The hi-hat is played in eighth notes. The bass plays the root notes of the chords. There is no percussion in this song. This song can be played in a romantic movie.

[10:00-20:00]
This song features a male voice singing the main melody. This is accompanied by programmed percussion playing a simple beat. The hi-hat is played in eighth notes. A synth plays chords in the background. There is no percussion in this song. This song can be played in a romantic movie.

[20:00-30:00]
This song features a male voice singing the main melody. This is accompanied by a guitar playing arpeggiated chords. A wind instrument plays fills in between lines. There is no percussion in this song. This song can be played in a romantic movie."""

parsed = parse_caption(sample)
print("=== full_text ===")
print(parsed['full_text'])
print("\n=== unique_sentences ===")
print(parsed['unique_sentences'])

=== full_text ===
This song features a male voice singing the main melody. This is accompanied by the percussion playing a simple beat. The hi-hat is played in eighth notes. The bass plays the root notes of the chords. There is no percussion in this song. This song can be played in a romantic movie. 
 
This song features a male voice singing the main melody. This is accompanied by programmed percussion playing a simple beat. The hi-hat is played in eighth notes. A synth plays chords in the background. There is no percussion in this song. This song can be played in a romantic movie. 
 
This song features a male voice singing the main melody. This is accompanied by a guitar playing arpeggiated chords. A wind instrument plays fills in between lines. There is no percussion in this song. This song can be played in a romantic movie.

=== unique_sentences ===
This song features a male voice singing the main melody. This is accompanied by the percussion playing a simple beat. The hi-hat is pla

In [ ]:
SAVE_PATH = "/content/drive/MyDrive/captions.csv"

captions_df = generate_captions_batch(
    df=df,
    client=client,
    audio_col='audio',
    save_path=SAVE_PATH
)

print(f"Готово! Описаний: {len(captions_df)}")
print(captions_df.head())

Генерация описаний: 100%|██████████| 1230/1230 [2:09:14<00:00,  6.30s/it]

Готово! Описаний: 1230
   idx                                            caption
0    0  [0:00-10:00]\nThis is a hip-hop music piece. T...
1    1  [0:00-10:00]\nThis is a hip-hop music piece. T...
2    2  [0:00-10:00]\nThe low quality recording featur...
3    3  [0:00-10:00]\nThe low quality recording featur...
4    4  [0:00-10:00]\nThe low quality recording featur...
